In [1]:
import json
import numpy as np
import pandas as pd
import hashlib

In [2]:
# TODO: Please verify dataset consistency across team members!
# Each team member should run this cell after generating expw_subset_metadata_comp90051.csv.
# Please compare your SHA256 hash with the reference hash below. The hashes should be identical.
# SHA256: deb6fa9592d87248968d7c5b67f1867ad5b589c8f2dfe3aa3ec450917461dfec
# If they differ, then the dataset contents or row order may be different between us, meaning the shared CV splits are invalid.

In [3]:
seed=90051
n_splits=10 #10 folds
# Use the subset dataset which created by data_preparation.ipynb
csv_path = "expw_subset_metadata_comp90051.csv"
df = pd.read_csv(csv_path)
correct_facial_class = df["correct_facial_class"].to_numpy()

In [4]:
#check the first and last rows
print("Total samples:", len(df),"\nFirst 5 rows:",df.head(),"\nLast 5 rows:",df.tail())

Total samples: 14935 
First 5 rows:   sample_face_id     original_image_name  face_id  top  left  right  bottom  \
0     expw_00000     afraid_black_76.jpg        0   55   244    377     188   
1     expw_00001  afraid_manager_398.jpg        0   18   311    424     131   
2     expw_00002    afraid_woman_295.jpg        0   11   308    444     147   
3     expw_00003   afraid_worker_569.jpg        0  108   378    702     432   
4     expw_00004    amazed_actor_142.jpg        0    0    46    326     280   

   width  height  confidence  correct_facial_class  
0    133     133     75.3854                     0  
1    113     113     85.8589                     0  
2    136     136     87.5403                     0  
3    324     324     44.2332                     0  
4    280     280     36.8803                     0   
Last 5 rows:       sample_face_id           original_image_name  face_id  top  left  right  \
14930     expw_14930  surprised_expression_497.jpg        0   57   439    66

In [5]:
# Calculate SHA value
canonical_hash = hashlib.sha256(df.to_csv(index=False, lineterminator="\n").encode("utf-8")).hexdigest()
print("Canonical SHA256:", canonical_hash)

Canonical SHA256: deb6fa9592d87248968d7c5b67f1867ad5b589c8f2dfe3aa3ec450917461dfec


In [6]:
def k_fold_CV_function(correct_facial_class,n_splits,seed):
    start = 0
    folds_list = [[] for _ in range(n_splits)]
    random_generator = np.random.default_rng(seed)
    correct_class_narray = np.asarray(correct_facial_class)
    for facial_class in np.unique(correct_class_narray):
        f_class_idxs = np.where(correct_class_narray == facial_class)[0]
        random_generator.shuffle(f_class_idxs)
        for i in range(len(f_class_idxs)):
            fold_num = (start + i) % n_splits
            folds_list[fold_num].append(int(f_class_idxs[i]))
        start = (start + len(f_class_idxs)) % n_splits
    result = []
    for fold in folds_list:
        result.append(np.array(sorted(fold), dtype=int))
    return result

In [7]:
# Implement outer 10-fold CV
start_fold_id=1
cv_splits_for_test_train_idxes = []
outer_10fold_CV_idx_lits =k_fold_CV_function(correct_facial_class,n_splits,seed)
all_df_idxes = np.arange(len(df))
for fold_id, test_idxes in enumerate(outer_10fold_CV_idx_lits, start_fold_id):
    train_idxes = np.setdiff1d(all_df_idxes, test_idxes)
    cv_splits_for_test_train_idxes.append({"fold number": fold_id,"train indexes": train_idxes.tolist(),"test indexes": test_idxes.tolist()})
    print(f"Fold {fold_id}: "f"Train = {len(train_idxes)}, "f"Test = {len(test_idxes)}")

Fold 1: Train = 13441, Test = 1494
Fold 2: Train = 13441, Test = 1494
Fold 3: Train = 13441, Test = 1494
Fold 4: Train = 13441, Test = 1494
Fold 5: Train = 13441, Test = 1494
Fold 6: Train = 13442, Test = 1493
Fold 7: Train = 13442, Test = 1493
Fold 8: Train = 13442, Test = 1493
Fold 9: Train = 13442, Test = 1493
Fold 10: Train = 13442, Test = 1493


In [8]:
# Save the 10 splits into json file
with open("Outer_10-fold_CV_splits.json", "w") as f:
    json.dump(cv_splits_for_test_train_idxes, f, indent=2)
print(f"Saved {len(cv_splits_for_test_train_idxes )} outer CV folds",f"\nTotal samples: {len(df)}")

Saved 10 outer CV folds 
Total samples: 14935
